In [ ]:
import subprocess, sys, os, time
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr)[-1500:], flush=True); return r.returncode

sh(f"{sys.executable} -m pip install -q -U transformers peft accelerate torchao bitsandbytes pycocoevalcap 'trl==0.29.1' 2>&1 | tail -3")
sh(f'{sys.executable} -c "import trl, transformers, peft, torch; print(trl.__version__, transformers.__version__, peft.__version__, torch.__version__, torch.cuda.get_device_name(0))"')
sh(f'{sys.executable} -m pip uninstall -y torchao 2>&1 | tail -1')


In [ ]:
import os, glob, json, shutil, hashlib
W = "/kaggle/working"
md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()

# dataset script: chỉ nhận thư mục có đủ 4 tệp (loại grpo_spice.py nằm trong output train)
SRC = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/gen_test_grpo.py", recursive=True)
       if all(os.path.exists(os.path.join(os.path.dirname(p), f))
              for f in ("grpo_spice.py", "build_branch_data.py", "preds_s1_seed101.jsonl"))]
assert len(SRC) == 1, f"DỪNG: cần đúng một grpo-spice-script bản mới, thấy {SRC}"
SRC = SRC[0]
for f in ("grpo_spice.py", "build_branch_data.py", "gen_test_grpo.py"):
    shutil.copy(f"{SRC}/{f}", f"{W}/{f}")
assert md5(f"{W}/grpo_spice.py") == "07ea87b6d156d1faa391a4274dffd7cf", "DỪNG: grpo_spice.py bản cũ"
assert md5(f"{W}/gen_test_grpo.py") == "b18ec1aa3fd037ddffb9411db63c2a3f", "DỪNG: gen_test_grpo.py lệch máy nhà"
S1P = f"{SRC}/preds_s1_seed101.jsonl"
assert sum(1 for _ in open(S1P)) == 6958

BUNDLE = next(r for r, d, f in os.walk("/kaggle/input") if "adapter_s1_seed101" in d)
MERGED = f"{W}/s1_merged"

def la_grpo(d):
    c = os.path.join(d, "adapter_config.json")
    return os.path.exists(c) and "s1_merged" in json.load(open(c)).get("base_model_name_or_path", "")
CK = [os.path.dirname(f) for f in glob.glob("/kaggle/input/**/adapter_model.safetensors", recursive=True)
      if la_grpo(os.path.dirname(f))]
assert len(CK) == 1, f"DỪNG: cần đúng một adapter GRPO, thấy {CK}"
CK500 = CK[0]
assert "grpo-spice-ck500-s202" in CK500 or "ck500-s202" in CK500, f"DỪNG: adapter không từ dataset s202: {CK500}"
assert md5(f"{CK500}/adapter_model.safetensors") == "57e5d2eec23278b0a2dfe7393d9fc463", "DỪNG: không phải adapter ck500/202 (s202)"

# thesis-score: gói mã cũ (score_run đã chấm S1 59,11) + test_ac đủ ảnh
best = None
for t in glob.glob("/kaggle/input/**/test_ac/test.jsonl", recursive=True):
    root = os.path.dirname(t)
    nimg = len(glob.glob(f"{root}/images/*.png"))
    if sum(1 for _ in open(t)) == 6958 and nimg >= 4463 and os.path.exists(f"{root}/ocr.jsonl"):
        best = root
assert best, "DỪNG: không thấy test_ac đủ 6.958 dòng + 4.463 ảnh"
TA = best
assert md5(f"{TA}/test.jsonl") == "da58299ee551a926a21cbecb5232bf78", "DỪNG: test.jsonl lệch máy nhà"
PKG = os.path.dirname(os.path.dirname(os.path.dirname(TA)))          # …/thesis/harness/dg1_cache/test_ac → …/thesis
shutil.copytree(f"{PKG}/harness", f"{W}/thesis/harness", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("images"))
link = f"{W}/thesis/harness/dg1_cache/test_ac/images"
if os.path.lexists(link): os.remove(link)
os.symlink(f"{TA}/images", link)
WS = f"{W}/thesis"

print("SRC   ", SRC); print("BUNDLE", BUNDLE); print("CK500 ", CK500); print("TA    ", TA)
print("có thư mục ref/ trong ck500:", os.path.isdir(f"{CK500}/ref"), flush=True)


In [ ]:
TEST = False

def chay(cmd, log, cwd, nhip=120, tran_gio=None):
    t0 = time.time()
    with open(log, "w") as f:
        q = subprocess.Popen(cmd, cwd=cwd, stdout=f, stderr=subprocess.STDOUT,
                             env={**os.environ, "TQDM_DISABLE": "1", "HF_HUB_DISABLE_PROGRESS_BARS": "1",
                                  "PYTHONUNBUFFERED": "1"})
        while q.poll() is None:
            time.sleep(nhip if not TEST else 30)
            L = [l.strip() for l in open(log, errors="ignore") if l.strip()]
            print(f"  {(time.time()-t0)/3600:5.2f} h · {os.path.basename(log)} · {L[-1][:110] if L else 'chưa có dòng nào'}", flush=True)
            if tran_gio and time.time() - t0 > tran_gio * 3600:
                q.terminate(); print(f"⛔ cắt ở {tran_gio} h để còn giờ chấm", flush=True); break
    print(f"--- {os.path.basename(log)} · mã thoát {q.returncode} · {(time.time()-t0)/3600:.2f} h ---")
    print("".join(open(log, errors="ignore").readlines()[-6:]), flush=True)
    return q.returncode

r = subprocess.run(["python", "grpo_spice.py", "--merge", "--bundle", BUNDLE, "--merged", MERGED],
                   cwd=W, capture_output=True, text=True)
print((r.stdout + r.stderr)[-1500:]); assert r.returncode == 0, "DỪNG: hoà lỗi"


In [ ]:
TA_ARGS = ["--recs", f"{TA}/test.jsonl", "--ocr", f"{TA}/ocr.jsonl", "--images", f"{TA}/images", "--tap-only", "--no-q4"]
out = f"{W}/kiem_hoa_test.jsonl"
if os.path.exists(out): os.remove(out)
chay(["python", "gen_test_grpo.py", "--bundle", BUNDLE, "--merged", MERGED, "--s1", S1P,
      "--n", "20", "--out", out] + TA_ARGS, f"{W}/kiem_hoa_test.log", W)
dong = [l for l in open(f"{W}/kiem_hoa_test.log") if "[kiểm hoà]" in l]
print(dong)
x = int(dong[-1].split("]")[1].split("/")[0])
assert x >= 16, f"DỪNG: kiểm hoà {x}/20 < 16 — đừng gắn điểm lưu, gửi log về"


In [ ]:
PRED = f"{W}/pred_ck500_s202_test{'_thu' if TEST else ''}.jsonl"
chay(["python", "gen_test_grpo.py", "--bundle", BUNDLE, "--merged", MERGED, "--ckpt", CK500,
      "--out", PRED] + (["--n", "5"] if TEST else []) + TA_ARGS,
     f"{W}/gen_test{'_thu' if TEST else ''}.log", W, tran_gio=6)
n = sum(1 for _ in open(PRED))
print(f"ck500/202 test: {n}/{5 if TEST else 4463} câu", flush=True)
assert n == (5 if TEST else 4463), "DỪNG: thiếu câu — chạy lại ô này (nối tiếp được), không xoá tệp"


In [ ]:
SC = f"{W}/score_ck500_s202_test{'_thu' if TEST else ''}.json"
cmd = ["python", "-u", "harness/score_run.py", "--mode", "score", "--grounder", "uground",
       "--preds", PRED, "--out", SC] + (["--n", "3"] if TEST else [])
chay(cmd, f"{W}/cham_test{'_thu' if TEST else ''}.log", WS)
R = [json.loads(l) for l in open(SC.replace(".json", "_raw.jsonl"))]
print(f"== ck500/202 test: {len(R)} bước · exec {sum(int(r.get('executable', 0)) for r in R)}", flush=True)
print("KIỂM (bản đủ): phải 4463 bước. Đừng tự diễn giải — gửi về máy nhà.")


In [ ]:
import zipfile, glob
fs = [f for f in glob.glob("/kaggle/working/*") if os.path.isfile(f) and not f.endswith(".zip") and "s1_merged" not in f
      and os.path.getsize(f) < 200*2**20]
with zipfile.ZipFile("/kaggle/working/ck500_s202_test.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in fs: z.write(f, os.path.basename(f))
print("đã zip", len(fs), "tệp:", sorted(os.path.basename(f) for f in fs), flush=True)
